In [ ]:
# import sys
# sys.path.append("/project01/ndcms/atownse2/ExponentialMixtureModel")

import ROOT
import numpy as np
import matplotlib.pyplot as plt

from tools import scale_out as so
import emm

In [ ]:
data_tree

In [ ]:
# Load the data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)

data_tree = emm.data.get_diphoton_data(tree=True)
data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))

In [ ]:
# Get the binning for the visualization
bins = emm.data.get_diphoton_binning()

In [ ]:
fit_options = [
    # ROOT.RooFit.IntegrateBins(0.0001),
    ROOT.RooFit.PrintLevel(-1),
    # ROOT.RooFit.Offset(True),
    # ROOT.RooFit.Strategy(2),
    ROOT.RooFit.Save(),
    # ROOT.RooFit.Range("fit_range")
]

In [ ]:
# Do model selection
model_primitives = [
    emm.make_model_primitive(emm.ExponentialMixtureModel, 1, data_mean=700, name="ExponentialMixture-1"),
    emm.make_model_primitive(emm.ExponentialMixtureModel, 2, data_mean=700, name="ExponentialMixture-2"),
    emm.make_model_primitive(emm.ExponentialMixtureModel, 3, data_mean=700, name="ExponentialMixture-3"),
    emm.make_model_primitive(emm.ExponentialMixtureModel, 4, data_mean=700, name="ExponentialMixture-4"),
]

results = {}

for mp in model_primitives:
    print(f"Fitting {mp.name}...")
    fit_result = emm.fit_random_restarts(
        x, data, mp,
        seed=42,
        n_samples=20, n_retries=5,
        save=False, fit_options=fit_options
    )
    print(f"   Number of minima found: {len(fit_result)}")
    print(f"   NLLs: {[res['nll'] for res in fit_result]}")

    model = mp(x) # Re-initialize the model to ensure it's in a clean state
    model.set_params(fit_result["final_pars"])

    aic = 2*len(model.params()) + 2*fit_result["nll"]
    bic = len(model.params())*np.log(data.numEntries()) + 2*fit_result["nll"]
    print(f"   Best fit: nll = {fit_result['nll']:.1f}")
    print(f"   AIC: {aic:.1f}, BIC: {bic:.1f}")
    results[model.n_components] = {
        "AIC": aic,
        "BIC": bic,
        "nll": fit_result["nll"],
    }


In [ ]:
criteria_by_k = {}
n_observations = data.numEntries()
for k, result in results.items():

    criteria_by_k[k] = emm.compute_information_criteria(
        result["nll"],
        2 * k - 1,
        n_observations,
    )

emm.plot_information_criteria(criteria_by_k)

In [ ]:
# Fit and plot
from emm.models import ModelPrimitive
model_primitives = [
    ModelPrimitive(emm.models.f1),
    ModelPrimitive(emm.models.f2),
    ModelPrimitive(emm.models.f3),
    ModelPrimitive(emm.models.f4),
    ModelPrimitive(emm.ExponentialMixtureModel, 2, data_mean=700, name="ExponentialMixture-2"),
]

# For visualization
from array import array
bin_array = array('d', bins)
binning = ROOT.RooBinning(len(bins) - 1, bin_array)
x.setBinning(binning)

# For chi2
hist_name = f"diphoton_binned_data_hist_{ROOT.TUUID().AsString().replace('-', '_')}"
hist = ROOT.TH1D(hist_name, "Binned Data", len(bins) - 1, bin_array)
hist.SetDirectory(0)
data.fillHistogram(hist, ROOT.RooArgList(x))

binned_data_name = f"binned_data_{ROOT.TUUID().AsString().replace('-', '_')}"
binned_data = ROOT.RooDataHist(
    binned_data_name,
    "Binned Data",
    ROOT.RooArgList(x),
    hist,
    )

models = []
model_labels = []
model_results = []
for model_primitive in model_primitives:
    print(f"Fitting model: {model_primitive.name}")
    fit_result = emm.fit_random_restarts(
        x, data, model_primitive,
        seed=42, n_samples=20, n_retries=5,
        save=False,
        fit_options=fit_options,
        print_level=2
        )


    model = model_primitive(x) # Re-initialize the model to ensure it's in a clean state
    model.set_params(fit_result["final_pars"])
    if "ExponentialMixture" in model.name:
        k = model.name.split("-")[-1]
        model_label = f"Exponential Mixture"
    else:
        model_func = "_".split(model.name)[0]
        model_num = model.name.split("_")[-1]
        model_label = f"#it{{f}}_{{{model_num}}}"

    nll = fit_result["nll"]
    n = data.numEntries()
    k = len(model.params())
    aic = 2*k + 2*nll
    bic = k*np.log(n) + 2*nll
    print(f"   AIC: {aic:.1f}, BIC: {bic:.1f}")

    chi2_res = emm.chi2(
        x,
        hist,
        model,
        min_events=30,
        print_chi2=True,
        )
    
    models.append(model)
    model_labels.append(model_label)
    model_results.append({
        "model": model,
        "label": model_label,
        "nll": nll,
        "aic": aic,
        "bic": bic,
        "chi2": chi2_res["chi2"],
        "chi2_ndf": chi2_res["chi2_ndf"],
    })

In [ ]:
from array import array
import importlib
importlib.reload(emm)


emm.plot_fits(
    data, x, models,
    labels=model_labels,
    # title="CMS High Mass Diphoton Spectrum",
    # colors=[ROOT.kRed],
    plot_range=(500, 1425),
    # nbins=128,
    y_min=5e-2,
    # logx=True,
    legend_bounds=(0.65, 0.56, 0.95, 0.85),
    legend_text_size=0.08,
    legend_columns=2,
    pull_fill_alpha=0.3
    )

In [ ]:
# Format results into a latex table
print("\n\nLaTeX Table:")
print("\\begin{tabular}{lcccc}")
print("Model & $\\chi^2$ & $\\chi^2_\\nu$ & $\Delta$AIC & $\Delta$BIC \\\\")
aic_min = min(res["aic"] for res in model_results)
bic_min = min(res["bic"] for res in model_results)
print("\\hline")
for res in model_results:
    delta_aic = res["aic"] - aic_min
    delta_bic = res["bic"] - bic_min
    label = res['label'].replace('#it{', '').replace('}', '').replace('{', '')
    if "Exponential Mixture" in label:
        label = "Exponential Mixture"
    else:
        label = f"${label}$"
    print(f"{label} & {res['chi2']:.1f} & {res['chi2_ndf']:.2f} & {delta_aic:.1f} & {delta_bic:.1f} \\\\")
print("\\end{tabular}")

In [ ]:
hist_name = f"diphoton_binned_data_hist_{ROOT.TUUID().AsString().replace('-', '_')}"
hist = ROOT.TH1D(hist_name, "Binned Data", len(bins) - 1, bin_array)
hist.SetDirectory(0)
data.fillHistogram(hist, ROOT.RooArgList(x))

binned_data_name = f"binned_data_{ROOT.TUUID().AsString().replace('-', '_')}"
binned_data = ROOT.RooDataHist(
    binned_data_name,
    "Binned Data",
    ROOT.RooArgList(x),
    hist,
    )
print(f"Number of bins in binned data: {binned_data.numEntries()}")

for model in models:
    print(f"Evaluating model: {model.name}")
    emm.chi2(
        x,
        hist,
        model,
        min_events=30,
        print_chi2=True,
        )